# SFT Flow


In [1]:
import sys
from pathlib import Path

ROOT = Path('/Users/montekkundan/Developer/ML/llm')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import copy
from course_tools import build_demo_bundle, format_messages, generate_text, train_model

LECTURE_NOTE_PATH = Path('/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/product/SFT Flow.md')
print(LECTURE_NOTE_PATH)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/product/SFT Flow.md


## Demo A: start from a base checkpoint


In [2]:
bundle = build_demo_bundle(steps=20)
base_model = bundle['model']
tokenizer = bundle['tokenizer']
prompt_messages = [
    {'role': 'system', 'content': 'You are concise.'},
    {'role': 'user', 'content': 'What is self-attention?'},
]
prompt = format_messages(prompt_messages, add_assistant_prompt=True)
base_text = generate_text(base_model, tokenizer, prompt, max_new_tokens=50, temperature=0.0)
print('base:', repr(base_text))


base: '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'


## Demo B: fine-tune on one canonical chat schema


In [3]:
sft_model = copy.deepcopy(base_model)
sft_corpus = '\n'.join([
    format_messages(prompt_messages + [{'role': 'assistant', 'content': 'Self-attention forms a weighted summary over relevant earlier tokens.'}]),
    format_messages([
        {'role': 'system', 'content': 'You are concise.'},
        {'role': 'user', 'content': 'What is tokenization?'},
        {'role': 'assistant', 'content': 'Tokenization maps text into discrete IDs.'},
    ]),
] * 6)
history = train_model(sft_model, tokenizer, train_text=sft_corpus, eval_text=sft_corpus, steps=12, learning_rate=2e-3, batch_size=4)
print(history)


[{'step': 1.0, 'train_loss': 6.040948390960693, 'eval_loss': 5.694355487823486, 'bpb': 8.215218423341797}, {'step': 2.0, 'train_loss': 5.824235916137695, 'eval_loss': 6.068273067474365, 'bpb': 8.754667461205326}, {'step': 4.0, 'train_loss': 5.0690155029296875, 'eval_loss': 5.00466251373291, 'bpb': 7.220201789885364}, {'step': 6.0, 'train_loss': 4.763462066650391, 'eval_loss': 4.656741142272949, 'bpb': 6.718257352660791}, {'step': 8.0, 'train_loss': 4.536865234375, 'eval_loss': 4.260495662689209, 'bpb': 6.14659596429066}, {'step': 10.0, 'train_loss': 4.415500164031982, 'eval_loss': 4.244089126586914, 'bpb': 6.122926336017713}, {'step': 12.0, 'train_loss': 3.9732699394226074, 'eval_loss': 3.7424488067626953, 'bpb': 5.399212334297359}]


## Demo C: compare the resulting checkpoint against the base model


In [4]:
sft_text = generate_text(sft_model, tokenizer, prompt, max_new_tokens=50, temperature=0.0)
print('sft :', repr(sft_text))


sft : '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'


## Rasbt and nanochat


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/gpt_instruction_finetuning.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/chat_sft.py
